In [ ]:
# Claim Statistics and Analytics Dashboard

This notebook builds a business-facing analytics dashboard for claim statistics using `output.csv`.
It calculates core KPIs, renders Plotly KPI cards, and provides dashboards for severity, issue type, and claim status.
</VSCode.Cell>
<VSCode.Cell language="python">
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

DATA_PATH = Path("..")  / "output.csv"
df = pd.read_csv(DATA_PATH)

df.head()
## Key Performance Indicators

Calculate total claims, valid/invalid counts, manual review counts, approval/rejection rates, and damage detection rate.
</VSCode.Cell>
<VSCode.Cell language="python">
# Define the core KPI fields
has_valid_image = "valid_image" in df.columns
has_damage_visible = "damage_visible" in df.columns
has_manual_review = "manual_review" in df.columns

# Compute base KPI values
kpis = {
    "total_claims": len(df),
    "valid_claims": 0,
    "invalid_claims": 0,
    "manual_review_claims": 0,
    "approval_rate": 0.0,
    "rejection_rate": 0.0,
    "damage_detection_rate": 0.0,
}

if has_valid_image:
    valid_mask = df["valid_image"].astype(str).str.lower().isin(["true", "1", "yes"])
    kpis["valid_claims"] = int(valid_mask.sum())
    kpis["invalid_claims"] = int((~valid_mask).sum())

if has_manual_review:
    manual_review_mask = df["manual_review"].astype(str).str.lower().isin(["true", "1", "yes"])
    kpis["manual_review_claims"] = int(manual_review_mask.sum())

if has_damage_visible:
    damage_mask = df["damage_visible"].astype(str).str.lower().isin(["true", "1", "yes"])
    kpis["damage_detection_rate"] = round(damage_mask.mean() * 100, 2)

if "claim_status" in df.columns:
    approval_mask = df["claim_status"].astype(str).str.lower().isin(["approved", "supported", "verified"])
    rejection_mask = df["claim_status"].astype(str).str.lower().isin(["rejected", "contradicted", "denied"])
    kpis["approval_rate"] = round((approval_mask.mean() * 100), 2)
    kpis["rejection_rate"] = round((rejection_mask.mean() * 100), 2)
else:
    # Fallback if claim_status is not available
    kpis["approval_rate"] = None
    kpis["rejection_rate"] = None

kpis
</VSCode.Cell>
<VSCode.Cell language="markdown">
## KPI Dashboard

Visualize the core claim statistics using Plotly KPI cards for a concise executive view.
</VSCode.Cell>
<VSCode.Cell language="python">
# KPI card formatting helper

def kpi_card(title, value, subtitle, color="#38bdf8"):
    return go.Figure(
        go.Indicator(
            mode="number+delta",
            value=value,
            title={"text": title, "font": {"size": 16}},
            delta={"reference": 0, "position": "top", "valueformat": ".0f"},
            domain={"x": [0, 1], "y": [0, 1]},
            number={"font": {"size": 32, "color": color}},
        )
    ).update_layout(
        paper_bgcolor="#0b1220",
        plot_bgcolor="#0b1220",
        margin=dict(l=10, r=10, t=30, b=10),
        annotations=[
            dict(
                x=0.5,
                y=-0.18,
                xref="paper",
                yref="paper",
                text=subtitle,
                showarrow=False,
                font={"size": 12, "color": "#cbd5e1"},
            )
        ],
    )

cards = [
    ("Total Claims", kpis["total_claims"], "Total rows in dataset", "#38bdf8"),
    ("Valid Claims", kpis["valid_claims"], "Valid image evidence", "#22c55e"),
    ("Invalid Claims", kpis["invalid_claims"], "Potential issues or bad images", "#ef4444"),
    ("Manual Review", kpis["manual_review_claims"], "Claims flagged for extra review", "#f97316"),
]

kpi_figures = [kpi_card(title, value, subtitle, color) for title, value, subtitle, color in cards]

for fig in kpi_figures:
    fig.show()
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Approval / Rejection / Damage Detection Rates

Show business ratios and highlight the claim pipeline's acceptance and quality performance.
</VSCode.Cell>
<VSCode.Cell language="python">
rate_cards = []
if kpis["approval_rate"] is not None:
    rate_cards.append(("Approval Rate", kpis["approval_rate"], "% of approved claims", "#22c55e"))
if kpis["rejection_rate"] is not None:
    rate_cards.append(("Rejection Rate", kpis["rejection_rate"], "% of rejected claims", "#ef4444"))
if kpis["damage_detection_rate"] is not None:
    rate_cards.append(("Damage Detection Rate", kpis["damage_detection_rate"], "% of claims with visible damage", "#a78bfa"))

rate_figures = [kpi_card(title, value, subtitle, color) for title, value, subtitle, color in rate_cards]
for fig in rate_figures:
    fig.show()
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Severity Dashboard

Analyze severity distribution with bar and pie charts to show how claim severity is spread across the dataset.
</VSCode.Cell>
<VSCode.Cell language="python">
if "severity" in df.columns:
    severity_counts = df["severity"].fillna("Missing").value_counts().reset_index()
    severity_counts.columns = ["severity", "count"]

    fig = px.bar(
        severity_counts,
        x="severity",
        y="count",
        title="Severity Dashboard",
        template="plotly_dark",
        color="count",
        color_continuous_scale="Reds",
    )
    fig.update_layout(xaxis_title="Severity", yaxis_title="Count", height=520)
    fig.show()

    fig = px.pie(
        severity_counts,
        names="severity",
        values="count",
        title="Severity Share",
        template="plotly_dark",
    )
    fig.show()
else:
    print("Column 'severity' not found in dataset.")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Issue Type Dashboard

Visualize the distribution and relative share of issue types to understand which damage categories dominate.
</VSCode.Cell>
<VSCode.Cell language="python">
if "issue_type" in df.columns:
    issue_counts = df["issue_type"].fillna("Missing").value_counts().reset_index()
    issue_counts.columns = ["issue_type", "count"]

    fig = px.bar(
        issue_counts,
        x="issue_type",
        y="count",
        title="Issue Type Dashboard",
        template="plotly_dark",
        color="count",
        color_continuous_scale="Blues",
    )
    fig.update_layout(xaxis_title="Issue Type", yaxis_title="Count", height=520)
    fig.show()

    fig = px.pie(
        issue_counts,
        names="issue_type",
        values="count",
        title="Issue Type Share",
        template="plotly_dark",
    )
    fig.show()
else:
    print("Column 'issue_type' not found in dataset.")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Claim Status Dashboard

If a claim status field exists, visualize the outcome distribution. Otherwise infer status signals from validity and damage detection.
</VSCode.Cell>
<VSCode.Cell language="python">
if "claim_status" in df.columns:
    status_counts = df["claim_status"].fillna("Missing").value_counts().reset_index()
    status_counts.columns = ["claim_status", "count"]

    fig = px.bar(
        status_counts,
        x="claim_status",
        y="count",
        title="Claim Status Dashboard",
        template="plotly_dark",
        color="count",
        color_continuous_scale="Viridis",
    )
    fig.update_layout(xaxis_title="Claim Status", yaxis_title="Count", height=520)
    fig.show()

    fig = px.pie(
        status_counts,
        names="claim_status",
        values="count",
        title="Claim Status Share",
        template="plotly_dark",
    )
    fig.show()
else:
    inferred_status = df.copy()
    if has_valid_image and has_damage_visible:
        inferred_status["status_inferred"] = inferred_status.apply(
            lambda row: "Approved" if str(row["valid_image"]).lower() in ["true", "1", "yes"] and str(row["damage_visible"]).lower() in ["true", "1", "yes"] else "Review Needed",
            axis=1,
        )
        status_counts = inferred_status["status_inferred"].value_counts().reset_index()
        status_counts.columns = ["claim_status", "count"]

        fig = px.bar(
            status_counts,
            x="claim_status",
            y="count",
            title="Inferred Claim Status Dashboard",
            template="plotly_dark",
            color="count",
            color_continuous_scale="Viridis",
        )
        fig.update_layout(xaxis_title="Inferred Status", yaxis_title="Count", height=520)
        fig.show()

        fig = px.pie(
            status_counts,
            names="claim_status",
            values="count",
            title="Inferred Claim Status Share",
            template="plotly_dark",
        )
        fig.show()
    else:
        print("No claim_status field and insufficient data to infer status.")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Executive Summary

Summarize the most important claim statistics and business implications for stakeholders.
</VSCode.Cell>
<VSCode.Cell language="markdown">
- **Total Claims:** `{} total records`.
- **Valid Claims:** `{} claims` are considered valid based on the image quality signal.
- **Invalid Claims:** `{} claims` were flagged as invalid or low quality.
- **Manual Review Claims:** `{} claims` may require additional human inspection.
- **Approval Rate:** `{}%` of claims are approved or supported, if a status field exists.
- **Rejection Rate:** `{}%` of claims are rejected or contradicted, if a status field exists.
- **Damage Detection Rate:** `{}%` of claims show visible damage in the evidence.
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Business Recommendations

Provide actionable next steps for the claims verification and review process.
</VSCode.Cell>
<VSCode.Cell language="markdown">
- Focus manual review resources on claims with invalid images or no visible damage, as they are the highest risk for false approvals or incorrect denials.
- Prioritize improving image quality capture and labeling for claim sources that produce a low `valid_image` rate.
- Use the severity dashboard to identify high-severity issue types that need faster escalation.
- Monitor the approval and rejection ratios over time to detect drift or emerging fraud patterns.
- Expand training data for underrepresented issue types if the issue type dashboard shows strong imbalance.
</VSCode.Cell>
